# Pipeline Smoke Test

This notebook runs a deliberately tiny end-to-end pass through the forecasting and RL pipeline. 
It is meant to catch integration errors before starting the expensive full experiment.


In [1]:
%load_ext autoreload
%autoreload 2


## Imports And Paths


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "fyp_pipeline").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from fyp_pipeline.core_pipeline import CONFIG, configure_vast_ai, prepare_data, print_task_summary
from fyp_pipeline.experiment_runner import (
    run_experiment,
    build_cl_summary,
    print_and_save_comparison_tables,
)
from fyp_pipeline.trainers import LOGGER, compute_mase


Project root: /workspace/fyp_continuallearning_selfdistillation


## Runtime Setup


In [3]:
DATA_DIR = str(PROJECT_ROOT / "data" / "processed")
OUTPUT_DIR = str(PROJECT_ROOT / "outputs" / "smoke_test")

configure_vast_ai(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    require_gpu=False,  # set True on Vast.ai if you want to require CUDA
)


Runtime diagnostics

Python         : 3.12.13

PyTorch        : 2.3.1+cu121

CUDA available : True

GPU            : NVIDIA GeForce RTX 4090

...

CPU cores      : 144

Vast.ai        : NO

Active device  : CUDA

Precision      : 32

{'paths': {'demand_csv': '/workspace/fyp_continuallearning_selfdistillation/data/processed/demand_forecasting.csv',
  'rl_csv': '/workspace/fyp_continuallearning_selfdistillation/data/processed/rl_environment.csv',
  'checkpoints': '/workspace/fyp_continuallearning_selfdistillation/outputs/smoke_test/checkpoints',
  'results': '/workspace/fyp_continuallearning_selfdistillation/outputs/smoke_test/results',
  'logs': '/workspace/fyp_continuallearning_selfdistillation/outputs/smoke_test/logs',
  'plots': '/workspace/fyp_continuallearning_selfdistillation/outputs/smoke_test/plots'},
 'tasks': [{'task_id': 1,
   'name': 'Baseline_2023_H1',
   'start': '2023-01-01',
   'end': '2023-05-31',
   'regime': 'baseline'},
  {'task_id': 2,
   'name': 'MegaSale_2023',
   'start': '2023-06-01',
   'end': '2023-12-31',
   'regime': 'mega_sale'},
  {'task_id': 3,
   'name': 'Baseline_2024_H1',
   'start': '2024-01-01',
   'end': '2024-05-31',
   'regime': 'baseline'},
  {'task_id': 4,
   'name': 'MegaSa

## Tiny Smoke-Test Configuration


In [4]:
# Keep this tiny. The goal is correctness, not final metrics.
CONFIG["tasks"] = CONFIG["tasks"][:2]
CONFIG["model_types"] = ["forecasting", "rl"]
CONFIG["cl_methods"] = {
    "forecasting": ["naive", "ewc", "replay", "sdft"],
    "rl": ["naive", "ewc", "recall", "sdft"],
}

CONFIG["forecasting"].update({
    "encoder_length": 28,
    "prediction_length": 7,
    "hidden_size": 16,
    "attention_head_size": 1,
    "hidden_continuous_size": 8,
    "batch_size": 64,
    "max_epochs": 1,
    "early_stop_patience": 1,
})

CONFIG["rl"].update({
    "total_timesteps_per_task": 256,
    "eval_episodes": 1,
    "n_steps": 128,
    "batch_size": 64,
    "n_epochs": 1,
    "net_arch": [32, 32],
})

CONFIG["cl"].update({
    "ewc_fisher_samples": 2,
    "replay_buffer_size": 128,
    "recall_buffer_capacity": 256,
    "recall_mix_n_steps": 32,
})

CONFIG["hardware"].update({
    "compile": False,
    "num_workers": 0,
    "persistent_workers": False,
})

print("Smoke-test config ready")
print("Tasks:", [t["name"] for t in CONFIG["tasks"]])
print("Forecast methods:", CONFIG["cl_methods"]["forecasting"])
print("RL methods:", CONFIG["cl_methods"]["rl"])


Smoke-test config ready
Tasks: ['Baseline_2023_H1', 'MegaSale_2023']
Forecast methods: ['naive', 'ewc', 'replay', 'sdft']
RL methods: ['naive', 'ewc', 'recall', 'sdft']


## Metric Sanity Check


In [5]:
mase_value = compute_mase([2, 3, 4], [2, 2, 5], list(range(20)), seasonality=7)
assert mase_value == mase_value and mase_value > 0, mase_value
print("MASE sanity check:", mase_value)


MASE sanity check: 0.09523809523809523


## Load Data


In [6]:
tft_tasks, rl_tasks, tft_df, rl_df = prepare_data()
print_task_summary(tft_tasks, rl_tasks)

assert len(tft_tasks) == len(CONFIG["tasks"])
assert len(rl_tasks) == len(CONFIG["tasks"])
assert all(len(df) > 0 for df in tft_tasks), "At least one TFT task is empty"
assert all(len(df) > 0 for df in rl_tasks), "At least one RL task is empty"
print("Data checks passed")


Loading datasets...

Demand CSV  : 9,864 rows × 66 cols

RL CSV      : 9,864 rows × 86 cols

✓ Data loaded and cleaned

┏━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┓
┃ Task ┃ Name             ┃ Period                   ┃ TFT rows ┃ RL rows ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━┩
│ 1    │ Baseline_2023_H1 │ 2023-01-01 -> 2023-05-31 │ 1,359    │ 1,359   │
│ 2    │ MegaSale_2023    │ 2023-06-01 -> 2023-12-31 │ 1,926    │ 1,926   │
└──────┴──────────────────┴──────────────────────────┴──────────┴─────────┘

Data checks passed


## Run Smoke Test


In [ ]:
run_experiment(tft_tasks, rl_tasks)
print("Smoke-test training loop completed")


==============================================================

  CONTINUAL LEARNING EXPERIMENT START

==============================================================

═══ MODEL TYPE: FORECASTING ═══

  ── CL Method: naive ──

...

Task 1/2: Baseline_2023_H1

Output()

Fisher computed over 2 batches

✓ Training complete

Evaluating on 1 seen task(s)...

## Validate Results


In [ ]:
results_df = LOGGER.to_dataframe()
display(results_df.tail(20))

assert not results_df.empty, "No metrics were logged"
expected_model_types = set(CONFIG["model_types"])
assert expected_model_types.issubset(set(results_df["model_type"])), results_df["model_type"].unique()

forecast_df = results_df[results_df["model_type"] == "forecasting"]
rl_df_results = results_df[results_df["model_type"] == "rl"]
assert not forecast_df.empty, "No forecasting metrics logged"
assert not rl_df_results.empty, "No RL metrics logged"

# MASE can be NaN if a deliberately tiny smoke split has insufficient scale,
# but sMAPE/RMSE and RL metrics should exist.
assert {"smape", "rmse"}.issubset(set(forecast_df["metric_name"])), forecast_df["metric_name"].unique()
assert "cumulative_profit" in set(rl_df_results["metric_name"]), rl_df_results["metric_name"].unique()

print("Logged metrics:")
print(results_df.groupby(["model_type", "cl_method", "metric_name"]).size())
print("Smoke test passed")


## Optional Summary Tables


In [ ]:
cl_summary = build_cl_summary()
tables = print_and_save_comparison_tables(cl_summary)
cl_summary
